In [1]:
import xml.etree.ElementTree as ET

# Ruta al XML
xml_path = "video01/vehicles.xml"   # ajustalo si está en otro lado

# Cargar XML
tree = ET.parse(xml_path)
root = tree.getroot()

# Encontrar el nodo <gtruth>
gtruth = root.find("gtruth")

# Obtener la lista de vehículos
vehicles = gtruth.findall("vehicle")

print(f"Total vehículos en el XML: {len(vehicles)}\n")

# Imprimir los primeros 3 vehículos
for i, v in enumerate(vehicles[:3], start=1):
    print(f"Vehículo #{i}")

    # Atributos del vehículo
    print("  Atributos:", v.attrib)

    # Región (patente)
    region = v.find("region")
    if region is not None:
        print("  Región:", region.attrib)

    # Radar (si existe)
    radar = v.find("radar")
    if radar is not None:
        print("  Radar:", radar.attrib)

    print()


Total vehículos en el XML: 119

Vehículo #1
  Atributos: {'iframe': '58', 'lane': '1', 'moto': 'True', 'plate': 'True', 'radar': 'False', 'sema': 'False'}
  Región: {'h': '41', 'w': '52', 'x': '493', 'y': '696'}

Vehículo #2
  Atributos: {'iframe': '71', 'lane': '3', 'moto': 'False', 'plate': 'True', 'radar': 'True', 'sema': 'False'}
  Región: {'h': '35', 'w': '106', 'x': '1632', 'y': '930'}
  Radar: {'frame_end': '111', 'frame_start': '71', 'speed': '56.65'}

Vehículo #3
  Atributos: {'iframe': '107', 'lane': '3', 'moto': 'False', 'plate': 'True', 'radar': 'True', 'sema': 'False'}
  Región: {'h': '27', 'w': '103', 'x': '1563', 'y': '896'}
  Radar: {'frame_end': '147', 'frame_start': '107', 'speed': '53.74'}



eliminamos los que tienen Radar = FALSE porque no tienen metrica de velocidad


In [2]:


xml_in = "video01/vehicles.xml"
xml_out = "video01/vehicles_with_speed.xml"

tree = ET.parse(xml_in)
root = tree.getroot()

gtruth = root.find("gtruth")
vehicles = gtruth.findall("vehicle")

removed = 0
for v in vehicles:
    radar_flag = v.get("radar")

    # Si es None o si es "false" → eliminar
    if radar_flag is None or radar_flag.lower() == "false":
        gtruth.remove(v)
        removed += 1

tree.write(xml_out)

print(f"Se eliminaron {removed} vehículos sin radar.")
print(f"Archivo guardado en: {xml_out}")


Se eliminaron 9 vehículos sin radar.
Archivo guardado en: video01/vehicles_with_speed.xml


asignamos IDs, quiero que el mismo auto tenga el mismo id en ambos datasets.

In [3]:

xml_in = "video01/vehicles_with_speed.xml"
xml_out_all = "video01/vehicles_with_speed.xml"

tree = ET.parse(xml_in)
root = tree.getroot()
gtruth = root.find("gtruth")

# Asignar IDs 1..N a todos los vehículos con velocidad
for v_id, v in enumerate(gtruth.findall("vehicle"), start=1):
    v.set("vehicle_id", str(v_id))

tree.write(xml_out_all)
print("IDs asignados. Archivo padre con IDs:", xml_out_all)


IDs asignados. Archivo padre con IDs: video01/vehicles_with_speed.xml


por las dudas, armamos el dataset para los autos a los que no los agarro el semaforo. hay una velocidad en varios, pero al haber frenado quizas es ruido para el calculo

In [4]:
import xml.etree.ElementTree as ET

# Archivo original
xml_in = "video01/vehicles_with_speed.xml"

# Archivo de salida
xml_out = "video01/vehicles_no_sema.xml"

# Cargar XML
tree = ET.parse(xml_in)
root = tree.getroot()

gtruth = root.find("gtruth")
vehicles = gtruth.findall("vehicle")

# Eliminar vehículos con sema="True"
removed = 0
for v in vehicles:
    sema_flag = v.get("sema")
    if sema_flag is not None and sema_flag.lower() == "true":
        gtruth.remove(v)
        removed += 1

# Guardar resultado
tree.write(xml_out)

print(f"Se eliminaron {removed} vehículos con sema=True.")
print(f"Archivo guardado en: {xml_out}")


Se eliminaron 6 vehículos con sema=True.
Archivo guardado en: video01/vehicles_no_sema.xml


In [5]:
#cantidad de autos en el xml
import xml.etree.ElementTree as ET
xml_in = "video01/vehicles.xml"
tree = ET.parse(xml_in)
root = tree.getroot()
gtruth = root.find('gtruth')
vehicles = gtruth.findall('vehicle')
print(f"Total vehículos en el XML raw: {len(vehicles)}")

xml_in = "video01/vehicles_with_speed.xml"
tree = ET.parse(xml_in)
root = tree.getroot()
gtruth = root.find('gtruth')
vehicles = gtruth.findall('vehicle')
print(f"Total vehículos en el XML con velocidad: {len(vehicles)}")





xml_in = "video01/vehicles_no_sema.xml"
tree = ET.parse(xml_in)
root = tree.getroot()
gtruth = root.find('gtruth')
vehicles = gtruth.findall('vehicle')
print(f"Total vehículos en el XML con velocidad y sin semaforo: {len(vehicles)}")


Total vehículos en el XML raw: 119
Total vehículos en el XML con velocidad: 110
Total vehículos en el XML con velocidad y sin semaforo: 104
